# End of week Exercise - week 2



In [ ]:
import os # for environment variables 
import json # for parsing tool calls
import tempfile # for storing tool call results
import mimetypes # for determining media types of tool call results
from pathlib import Path # for handling file paths
import gradio as gr # for UI
from dotenv import load_dotenv # for loading environment variables from .env file
from openai import OpenAI # for interacting with the OpenAI API

load_dotenv(override=True) # override=True allows ev in the .env file to overwrite existing ev

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
gemma_url = "http://localhost:11434/v1"
ollama_url = "http://localhost:11434/v1"



gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=gemini_url)
groq = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
gemma = OpenAI(api_key="ollama", base_url=gemma_url)
cohere = OpenAI(base_url="https://api.cohere.ai/compatibility/v1", api_key=os.getenv("COHERE_API_KEY"))

MODEL_OPTIONS = {
"GEMINI":(gemini,"emini-2.5-flash-lite"),
"GROQ":(groq,"openai/gpt-oss-20b"),
"OLLAMA":(ollama,"llama3:8b"),
"GEMMA":(gemma,"gemma3:270m"),
"COHERE":(cohere,"command-a-03-2025"),
}

In [ ]:
SYSTEM_PROMPT = """
You are a senior software engineer and technical tutor.
Explain concepts clearly with practical examples.
Keep answer concise but useful.
If relevant, provide one short check-your-understanding question.
If the user asks for definition of a known term, use the glossary tool.
"""

GLOSSARY = {
    "api": "Application Programming Interface - a set of rules that allows different software applications to communicate with each other.",
    "token": "In the context of language models, a token is a unit of text that the model processes. It can be as short as one character or as long as one word, depending on the language and the model's tokenization method.",
    "latency": "The time it takes for a model to process a request and return a response.",
    "vector database": "A type of database optimized for storing and querying high-dimensional vectors, often used for tasks like similarity search in machine learning applications.",
    "rag": "Retrieval-Augmented Generation - a technique that combines retrieval of relevant information with generation of responses, often used to enhance the capabilities of language models by providing them with access to external knowledge sources.",
    "langchain": "A framework for developing applications powered by language models, providing tools and abstractions to manage interactions with models, handle prompts, and integrate with various data sources and APIs."
}

# for tool calls
glossary_tool = {
    "type"  : "function",
    "function":{
        "name":"lookup_glossary",
        "description":"Looks up a term in the glossary. Input is a JSON object with a 'term' field. Output is a string definition of the term.",
        "parameters":{
            "type":"object",
            "properties":{
                "term":{
                    "type":"string",
                    "description":"The term to look up in the glossary"
                }
            },
            "required":["term"],
            "additionalProperties":False
        }
    }
}

In [ ]:
SYSTEM_PROMPT += """
topics other then software engineering are out of scope. If the user asks about a term not in the glossary, respond with "Term not found in glossary.
and tell the user to ask about another term related to software engineering in a friendly way.
"""

In [ ]:
def lookup_glossary(term:str) ->str:
    key = term.lower()
    return GLOSSARY.get(key, f"{term} not found in glossary.")

def handle_tool_calls(tool_calls):
    response = []
    for call in tool_calls:
        name = call.function.name
        args = json.loads(call.function.arguments or "{}")

        if name == "lookup_glossary":
            result = lookup_glossary(args.get("term", ""))
        else:
            result = f"Unknown tool: {name}"
        
        response.append({
            "role":"tool",
            "tool_call_id":call.id,
            "content":result
            })
    return response

# Transcription
def transcribe_audio(audio_path:str) -> str:

    if not audio_path:
        return ""
    # we need to read the audio file as bytes and send it to the model. The model will return a transcription of the audio.
    with open(audio_path,"rb") as f:
        result = groq.audio.transcriptions.create(file=f, model="whisper-large-v3-turbo")
        # getattr is used to safely get the "text" attribute from the result. If it doesn't exist, it returns an empty string.
    return (getattr(result, "text", "") or "").strip()

# Text-to-Speech
def talker (text:str) -> str|None:
    if not text.strip():
        return None
    response = groq.audio.speech.create(
            input=text,
            model="canopylabs/orpheus-v1-english",
            response_format= "wav",
            voice="austin"
    )

    # the response from the model is in bytes, we need to write it to a temporary file and return the file path. We use tempfile.NamedTemporaryFile to create a temporary file that will be automatically deleted when closed. We write the response content to the file and return the file name.
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    tmp.write(response.content)
    tmp.close()
    return tmp.name


In [ ]:
def add_user_turn(user_text,audio_path,history):
    history = history or []

    text = (user_text or "").strip()
    if not text and audio_path:
        text = transcribe_audio(audio_path)
    
    if not text:
        return "",None, history, "Please enter some text or record audio to send a message."
    
    # We create a new message with the role "user" and the content of the text, and append it to the history.
    history.append({"role":"user", "content":text})
    return "",None, history, None

def build_message(history):
    messages = [{"role":"system", "content":SYSTEM_PROMPT}]
    for turn in history:
        messages.append({"role":turn["role"], "content":turn["content"]})
    return messages

# This function handles the assistant's response generation, including tool calls and voice output.
def assistant_stream(history,model_choice,use_tools,voice_enabled):
    history = history or []
    client, model = MODEL_OPTIONS.get(model_choice)
    message = build_message(history)

    # step 1 - generate response with tool calls if needed
    while True:
        response = client.chat.completions.create(
            model=model,
            messages=message,
            tools=[glossary_tool] if use_tools else None,            
        )

        msg = response.choices[0].message
        if response.choices[0].finish_reason == "tool_calls":
            message.append({
                "role":"assistant",
                "content":msg.content or "",
                "tool_calls":[
                    {
                        "id": tc.id,
                        "type":  "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in msg.tool_calls
                ]
            })
            message.extend(handle_tool_calls(msg.tool_calls))
            continue 

        final_text = (msg.content or "").strip()
        break

    # step 2 - update history with the final response
    history.append({
        "role":"assistant",
        "content":""
    })

    words = final_text.split()
    chunk = ""
    for word in words:
        chunk = (chunk + " " + word).strip()
        history[-1]["content"] = chunk
        yield history, None

    # step 3 - generate voice response if enabled
    if voice_enabled:
        audio_file = talker(final_text)
        yield history, audio_file
    else:
        yield history, None

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("Tech Tutor - A multi-modal chatbot for learning technical concepts.")
    gr.Markdown(
        "Enter your question or topic below. You can also record audio by clicking the microphone icon. "
        "The assistant will respond with text and optionally audio."
    )

    with gr.Row():
        model_choice = gr.Dropdown(
            label="Model",
            choices=list(MODEL_OPTIONS.keys()),
            value="GROQ"
        )
        use_tools = gr.Checkbox(label="Enable glossary tool", value=True)
        voice_enabled = gr.Checkbox(label="Enable voice response", value=False)

    chatbot = gr.Chatbot(type="messages", label="Tech Tutor", height=420)

    with gr.Row():
        user_text = gr.Textbox(
            label="Your question or topic",
            placeholder="Ask me anything about software engineering...",
            scale=4
        )
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="Or record your question/topic",
            scale=2
        )

    # interactive=False so that the user cannot edit it.
    error_box = gr.Textbox(label="Error", interactive=False)
    audio_output = gr.Audio(label="Assistant response (audio)", autoplay=True)
    send_button = gr.Button("Send", variant="primary")
    clear_button = gr.Button("Clear", variant="secondary")

    send_event = send_button.click(
        add_user_turn,
        inputs=[user_text, audio_input, chatbot],
        outputs=[user_text, audio_input, chatbot, error_box]
    )
    send_event.then(
        assistant_stream,
        inputs=[chatbot, model_choice, use_tools, voice_enabled],
        outputs=[chatbot, audio_output]
    )

    clear_button.click(
        # reset the interface to the initial state
        lambda: ("", None, [], ""),
        outputs=[user_text, audio_input, chatbot, error_box]
    )

demo.launch(inbrowser=True)